# 给 LLM API Key 加密

本场比赛的因子代码里如果要调用 LLM，**请不要把 `api_key` 明文写进 notebook**。

正确做法：用平台公钥把你的 key 加密成一段密文，把**密文**填到提交页的「LLM 密钥」字段。
平台在评估时会用私钥解密，并通过环境变量 `LLM_API_KEY` 注入到你的代码里，全程只有平台的评估环境能拿到明文。

你的代码里取 key 只需：
```python
import os
api_key = os.environ.get("LLM_API_KEY")   # 平台注入，不要硬编码
```

运行本 notebook 前，确保同目录下有平台发的公钥 `llm_key_public.pem`，并安装依赖：
```bash
pip install cryptography
```

In [ ]:
# ==== 在这里填你自己的 LLM API Key（明文只在你本机出现，不会上传）====
MY_API_KEY = "sk-在这里粘贴你的key"

# 平台公钥文件（和本 notebook 放同一目录）
PUBLIC_KEY_PATH = "llm_key_public.pem"

In [ ]:
import base64
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding

# 1) 读平台公钥
with open(PUBLIC_KEY_PATH, "rb") as f:
    public_key = serialization.load_pem_public_key(f.read())

# 2) 用公钥加密（RSA-OAEP / SHA-256），再转成 base64 文本
ciphertext = public_key.encrypt(
    MY_API_KEY.encode("utf-8"),
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)
encrypted_key = base64.b64encode(ciphertext).decode()

print("把下面这一整串密文填到提交页的「LLM 密钥」字段：\n")
print(encrypted_key)

## 说明

- 每次运行密文都不一样（RSA-OAEP 带随机填充），这是正常的，任选一次的输出即可。
- 密文只能用平台私钥解开，你和其他人都无法从密文反推出明文 key。
- 提交因子 notebook 时，记得删掉所有 `LLM_API_KEY` 硬编码，改用 `os.environ.get("LLM_API_KEY")`。